# Rating Model Comparison (v1 selection)

Compare candidate latent formulations (A weighted era z, B robust z, C weighted blended percentiles) and justify selecting **C** for v1. Production logic lives in `data-pipeline/rating/`.

In [ ]:
import sys; sys.path.insert(0, '../data-pipeline')
import numpy as np, pandas as pd
from rating.normalization import weighted_latent
from rating.versions import load_config

df = pd.read_parquet('../data/processed/player_tournament_stats.parquet')
cfg = load_config('v1')
odi_bat = df[(df.format=='ODI') & (df.bat_innings>0)].copy()
blend = cfg.batting['blend']; feats = cfg.batting['features']['ODI']

# Candidate C (chosen): weighted blended percentiles
odi_bat['C_percentile'] = weighted_latent(odi_bat, feats, blend)

# Candidate A: weighted era z-scores of the same raw metrics
def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)
za = np.zeros(len(odi_bat)); wsum = 0
for m, w in feats.items():
    za = za + w * zscore(odi_bat[m].fillna(odi_bat[m].median())).to_numpy(); wsum += w
odi_bat['A_weighted_z'] = za / wsum

# Candidate B: robust (median/IQR) z-scores
def robust_z(s):
    med = s.median(); iqr = s.quantile(0.75) - s.quantile(0.25)
    return (s - med) / (iqr if iqr else 1)
zb = np.zeros(len(odi_bat)); wsum = 0
for m, w in feats.items():
    zb = zb + w * robust_z(odi_bat[m].fillna(odi_bat[m].median())).to_numpy(); wsum += w
odi_bat['B_robust_z'] = zb / wsum

print('Rank correlation between candidates (ODI batting):')
print(odi_bat[['C_percentile','A_weighted_z','B_robust_z']].corr(method='spearman').round(3))
print('\nDistributions:')
print(odi_bat[['C_percentile','A_weighted_z','B_robust_z']].describe().round(2))
# C is chosen: bounded [0,100], interpretable, era-fair, robust to skew (see docs/rating-methodology.md).
